# Precision at fixed candidate budgets, containment-gated click-centred seeds (K = 10, 20, 30, 50)

**Question.** If the pathologist's candidate list is capped at 10, 20, 30 or 50 entries, what
fraction of what they read is actually a mitotic figure -- on each of the 14 decision-grade
ROIs in `images/extra_valid`, and per tumour domain? Precision only; `recall@K` is not reported
anywhere in this notebook (see the closing summary for where the recall-family columns that
`compare.evaluate_arms` computes as a byproduct end up instead).

**What this variant is.** A copy of `precision_at_k_budgets_14roi/precision_at_k_budgets_14roi.ipynb`
with exactly one change: the seed's template size comes from `seed_selection.tightened_base_size` --
the Otsu component the click's own pixel lands inside -- rather than the original's inline
`largest_cc_box`, which sized on the window's largest component whether or not the click was in it.
The template is still built centred on the raw click. That pair -- containment gate and size
correction kept, position correction dropped -- is what `DECISIONS.md` D8's "Amendment, 2026-09-09 --
recentring is reversed" settles on, and this notebook is that amendment run end to end.

**Scope, fixed for this run:**
- 14 ROIs, `images/extra_valid` (2 per tumour domain x 7 domains).
- 1 seed per ROI, `seed_index = 0`.
- **Seed construction: click-centred, containment-gated** (`seed_selection.tightened_base_size`),
  per `DECISIONS.md` D8's "Amendment, 2026-09-09 -- recentring is reversed". The template's *size*
  comes from the Otsu component the click's own pixel sits inside; its *centre* stays the raw
  click. A click outside its component is refused as a seed and redrawn on the same RNG stream.
  This replaces the source notebooks' inline `largest_cc_box`, which took the window's largest
  component with no containment check, and is **not** the recentred `tightened_template_box`
  variant the same amendment reverses.
- Today's decided defaults (`DECISIONS.md`): `TM_CCOEFF` (D1), no `tissue_mask` (D2),
  `hematoxylin_od` unclipped (D3), ranked by `tm_score` (D5), NMS radius = match radius =
  7.5 um (D7, enforced via `invariants.check_nms_radius`).
- No z-threshold sweep. Candidates are extracted once per ROI at a permissive deep floor
  (`z = -1.5`, the repo's standing "near-unfiltered" constant), NMS'd at 7.5 um, ranked by
  score, and then `compare.evaluate_arms`'s own budget loop truncates to K = 10/20/30/50. A
  sibling notebook (`find_and_suppress_high_threshold_precision.ipynb`, its Gate 2) already
  showed this is numerically identical to applying any higher threshold first, provided the
  threshold doesn't shrink the pool below K -- which the assertions below confirm holds on
  every ROI at every budget here. Each ROI's own z at rank 10/20/30/50 is reported as a
  diagnostic column, not applied as a filter.

In [1]:
import gc
import time
import sys

import cv2
import numpy as np
import pandas as pd

sys.path.insert(0, '..')
from midog_utils import channels as ch
from midog_utils import dataset as ds
from midog_utils import evaluate as ev
from midog_utils import find_and_suppress as fs
from midog_utils import seed_selection as ss
from midog_utils import template_match as tm
from midog_utils import compare as cp
from midog_utils import invariants as inv
from midog_utils.nms import nms_by_distance

pd.set_option('display.width', 220)
pd.set_option('display.max_columns', 60)
pd.set_option('display.max_rows', 220)

NB_T0 = time.time()

# ---------------------------------------------------------------------------------------
# Configuration. D1/D2/D3/D5/D7 per DECISIONS.md. No z-sweep -- one deep-floor extraction
# per ROI, then evaluate_arms's own budget loop does the K = 10/20/30/50 truncation.
# ---------------------------------------------------------------------------------------
IMAGES_DIR = '../images/extra_valid'
SEED_INDEX = 0                       # one seed per ROI

CHANNEL = 'hematoxylin_od'           # D3 -- unclipped optical density
METHOD = cv2.TM_CCOEFF               # D1 -- unnormalized, contrast-sensitive
PEAK_MIN_DISTANCE = 7
SELF_HIT_RADIUS = 5.0
DEEP_FLOOR_Z = -1.5                  # near-unfiltered extraction floor (repo convention)
MAX_PEAKS = 2_000_000

NMS_RADIUS_UM = ev.MIDOG_RADIUS_UM   # 7.5 um -- D7: NMS radius == match radius
MATCH_RADIUS_UM = ev.MIDOG_RADIUS_UM

CFG = fs.FSConfig(channel=CHANNEL, base_size=tm.BASE_SIZE, scales=(1.0,),
                  n_angles=1, flips=(False,), peak_min_distance=PEAK_MIN_DISTANCE,
                  self_hit_radius=SELF_HIT_RADIUS)
BORDER = CFG.patch_size // 2         # 36
OTSU_WINDOW = tm.BASE_SIZE           # 51

BUDGETS = (10, 20, 30, 50)           # the fixed candidate-list lengths under test

OUT_PER_ROI = '../results/precision_at_k_14roi_gatedseed_per_roi.csv'
OUT_BY_DOMAIN = '../results/precision_at_k_14roi_gatedseed_by_domain.csv'
OUT_RAW = '../results/precision_at_k_14roi_gatedseed_raw.csv'
OUT_VERIF = '../results/precision_at_k_14roi_gatedseed_verification.csv'

print(f'budgets {BUDGETS} x 14 ROIs x 1 click (seed_index={SEED_INDEX})')
print(f'NMS radius = match radius = {NMS_RADIUS_UM} um (D7)')

budgets (10, 20, 30, 50) x 14 ROIs x 1 click (seed_index=0)
NMS radius = match radius = 7.5 um (D7)


## The pipeline

Copied from `precision_at_k_budgets_14roi/precision_at_k_budgets_14roi.ipynb`, which copied it from
`find_and_suppress_high_threshold_precision.ipynb`, which copied it from
`recall_workload_ledger.py` -- the seed draw and the NMS+self-hit helper are kept inline rather than
promoted to `midog_utils`, for the same reason those notebooks give.

**Seed sizing is the one change in this notebook, and the reason it exists.** The source
notebooks size the seed's template with an inline `largest_cc_box`: the *largest* Otsu component
in the click's 51 px window, taken whether or not the click itself lands inside it. This one calls
`seed_selection.tightened_base_size` instead, which routes the seed through
`seed_selection.tighten_box_otsu`'s containment gate -- the component measured is the one the
click's own rounded pixel is foreground of (`center_tolerance=0`), and it must still clear the same
`min_area=50` / `max_area_frac=0.85` / `min_solidity=0.5` checks the inline helper applied. A click
that is not inside any accepted component returns `None`, which `draw_seed_with_retry` treats
exactly as it treated a failed gate before: drop that candidate, redraw on the same RNG stream,
count it in `n_retries`. `tightened_base_size` reads its own patch and returns an odd `base_size`
directly, so the manual `read_padded_patch`, the bbox unpacking and the local `_odd_local` are all
gone with the helper.

**Size only -- the click stays the centre.** `seed_xy` is the raw click, the template patch is read
there, and self-hit removal, the seed-annulus check and every ground-truth match are all referenced
to it, exactly as in the source notebooks. `DECISIONS.md` D8's "Amendment, 2026-09-09 -- recentring
is reversed" keeps the containment gate and the size correction and rejects the position correction
outright ("size still comes from the accepted component, position reverts to the click"), so
`seed_selection.tightened_template_box` -- the recentred variant D8 originally decided on -- is
deliberately not called here, and no recentred point exists anywhere in this notebook.

In [2]:
def draw_seed_with_retry(pool, rng, check_fn):
    '''Draw a row via `rng.integers`; on failure drop it and redraw on the same stream.'''
    working, retries = pool.copy(), 0
    while len(working) > 0:
        idx = int(rng.integers(len(working)))
        row = working.iloc[idx]
        result = check_fn(row)
        if result is not None:
            return row, result, retries
        working = working.drop(working.index[idx])
        retries += 1
    raise ValueError('seed pool exhausted -- no candidate passed check_fn')


def suppress(centers, scores, radius, seed_xy):
    '''NMS at `radius`, then drop the seed's own self-correlation. Returns (centers, scores).'''
    keep = nms_by_distance(centers, scores, radius)
    c, s = centers[keep], scores[keep]
    if len(c):
        ok = np.hypot(c[:, 0] - seed_xy[0], c[:, 1] - seed_xy[1]) > SELF_HIT_RADIUS
        c, s = c[ok], s[ok]
    return c, s


def roi_files(images_dir=IMAGES_DIR):
    import os
    return sorted(f for f in os.listdir(images_dir) if f.endswith('.tiff'))


print(f'{len(roi_files())} ROIs on disk in {IMAGES_DIR}/')

14 ROIs on disk in ../images/extra_valid/


## The run

Per ROI: one `matchTemplate` pass (the expensive step, ~15 s), one extraction at the deep
floor, one NMS at 7.5 um. No re-run per budget -- `compare.evaluate_arms` computes
`tp_at_budget` / `budget_delivered` for every K in `BUDGETS` from that single ranked list.

In [3]:
def run_roi(fn, image_id, domain, anns):
    t0 = time.time()
    path = f'{IMAGES_DIR}/{fn}'
    rgb = ds.load_roi(path)
    mpp = ds.roi_mpp(path)
    roi_shape = rgb.shape
    match_radius = ev.radius_px(mpp, MATCH_RADIUS_UM)
    nms_radius = ev.radius_px(mpp, NMS_RADIUS_UM)
    hem = ch.to_channel(rgb, CHANNEL)
    gray_inv = ch.to_gray_inverted(rgb)
    H, W = hem.shape[:2]
    del rgb
    gc.collect()

    # --- the click -------------------------------------------------------------------
    gt = ds.image_annotations(anns, fn)
    seed_pool, flagged = ss.agreement_pool(gt[gt['category_id'] == ds.MITOTIC])
    seed_pool = ss.border_filter(seed_pool, BORDER, roi_shape)
    rng = np.random.default_rng([SEED_INDEX, image_id])

    # Containment-gated, size-only, click-centred -- DECISIONS.md D8's 2026-09-09 amendment.
    # `tightened_base_size` reads its own OTSU_WINDOW patch, runs `tighten_box_otsu` (the
    # click's own pixel must be foreground of a component clearing min_area/max_area_frac/
    # min_solidity), and returns that component's odd longer side. None -> this click is not
    # inside any accepted component; draw_seed_with_retry drops it and redraws on the same
    # RNG stream. The template centre is untouched: seed_xy below is still the raw click.
    def _check(row):
        return ss.tightened_base_size(gray_inv, float(row['cx']), float(row['cy']),
                                       otsu_window=OTSU_WINDOW)

    seed, base_size, n_retries = draw_seed_with_retry(seed_pool, rng, _check)
    seed_xy = (float(seed['cx']), float(seed['cy']))
    seed_ann_id = int(seed['ann_id'])
    gt_eval = gt[gt['ann_id'] != seed_ann_id].reset_index(drop=True)
    n_gt = int((gt_eval['category_id'] == ds.MITOTIC).sum())
    del gray_inv
    gc.collect()

    # --- one match, one deep-floor extraction, one NMS -- no z-sweep --------------------
    patch = tm.read_padded_patch(hem, *seed_xy, CFG.patch_size)
    templates, _ = tm.build_augmentations(patch, base_size, CFG.scales, CFG.n_angles, CFG.flips)
    PAD = max((t.shape[0] - 1) // 2 for t in templates)
    hem_p = cv2.copyMakeBorder(hem, PAD, PAD, PAD, PAD, borderType=cv2.BORDER_REPLICATE)
    fused_p, _, valid_p = tm.fused_response(hem_p, templates, CFG.scale_normalize, method=METHOD)
    fused = fused_p[PAD:PAD + H, PAD:PAD + W]
    valid = valid_p[PAD:PAD + H, PAD:PAD + W]
    assert bool(valid.all()), f'{fn}: padding left part of the ROI unreachable (PAD={PAD})'
    del hem_p, fused_p, valid_p, hem
    gc.collect()

    med, mad = tm.robust_stats(fused, valid)
    cut = med + DEEP_FLOOR_Z * mad
    centers, scores = tm.extract_peaks(fused, valid, PEAK_MIN_DISTANCE, cut, MAX_PEAKS)
    n_peaks = len(centers)
    assert n_peaks < MAX_PEAKS, f'{fn}: MAX_PEAKS is binding, raise it'
    c, s = suppress(centers, scores, nms_radius, seed_xy)
    assert bool(np.all(np.diff(s) <= 0)), f'{fn}: post-NMS pool is not score-descending'
    pool = pd.DataFrame({'cx': c[:, 0], 'cy': c[:, 1], 'score': s})

    # Seed annulus: at NMS radius == match radius, NMS itself empties this before self-hit
    # removal ever runs (the self-correlation is the map's global maximum, kept first, and
    # suppresses everything within one NMS radius) -- so this must count zero here.
    d_seed = np.hypot(pool['cx'] - seed_xy[0], pool['cy'] - seed_xy[1])
    n_near_seed = int((d_seed <= match_radius).sum())

    arm = cp.Arm('tm_score_deep_floor', (lambda d=pool: d), rank_key='score',
                 seeded=True, z=DEEP_FLOOR_Z, z_dependent=True,
                 nms_radius=nms_radius, caps=(MAX_PEAKS,))
    ctx = dict(file_name=fn, tumor_type=domain, image_id=image_id,
               seed_ann_id=seed_ann_id, base_size=base_size,
               map_median=round(float(med), 5), mad_scale=round(float(mad), 5), mpp=mpp)
    checks = []
    out = cp.evaluate_arms([arm], gt_eval, match_radius, roi_shape=(H, W), mpp=mpp,
                            budgets=BUDGETS, context=ctx, checks=checks)

    z_at_rank = {}
    for k in BUDGETS:
        z_at_rank[k] = (float(pool['score'].iloc[k - 1] - med) / mad) if len(pool) >= k else np.nan
    out['z_at_rank'] = out['budget'].map(z_at_rank)

    checks.append(dict(check='seed_annulus_empty', label=fn, n_near_seed=n_near_seed,
                       match_radius_px=round(match_radius, 3), passed=bool(n_near_seed == 0)))

    meta = dict(file_name=fn, tumor_type=domain, image_id=image_id, seed_ann_id=seed_ann_id,
                n_retries=n_retries, contested_seed_tier=bool(flagged), base_size=base_size,
                mpp=mpp, roi_h=int(H), roi_w=int(W), pad_px=PAD,
                match_radius_px=match_radius, nms_radius_px=nms_radius,
                map_median=float(med), mad_scale=float(mad), n_gt_mitotic=n_gt,
                n_detections=len(pool), n_near_seed_annulus=n_near_seed,
                t_total_s=round(time.time() - t0, 1))
    del fused, valid, templates, patch, centers, scores, c, s
    gc.collect()
    print(f"[{fn}] {domain:32s} base={base_size:2d} n_detections={len(pool):6d} "
          f"n_gt={n_gt:3d} [{meta['t_total_s']:.0f}s]", flush=True)
    return out, pd.DataFrame(checks), meta

In [4]:
images, annotations = ds.load_annotations('../databases/MIDOG++.json')
ds.check_invariants(annotations)
meta_ix = images.set_index('file_name')[['image_id', 'tumor_type']]

files = roi_files()
missing = sorted(set(files) - set(meta_ix.index))
assert not missing, f'.tiff on disk absent from the annotation DB: {missing}'
assert len(files) == 14, f'expected 14 ROIs in {IMAGES_DIR}/, found {len(files)}'

t_run = time.time()
out_frames, check_frames, roi_meta = [], [], []
for fn in files:
    image_id = int(meta_ix.loc[fn, 'image_id'])
    domain = meta_ix.loc[fn, 'tumor_type']
    out, checks, m = run_roi(fn, image_id, domain, annotations)
    out_frames.append(out)
    check_frames.append(checks)
    roi_meta.append(m)
    gc.collect()

RAW = pd.concat(out_frames, ignore_index=True)
VERIF = pd.concat(check_frames, ignore_index=True)
ROI = pd.DataFrame(roi_meta).set_index('file_name')

print(f'\n{len(files)} ROIs x {len(BUDGETS)} budgets = {len(RAW)} rows in {time.time() - t_run:.0f}s')
ROI[['tumor_type', 'seed_ann_id', 'base_size', 'n_gt_mitotic', 'n_detections',
     'match_radius_px', 'nms_radius_px', 'map_median', 'mad_scale', 't_total_s']].round(3)

[013.tiff] human breast cancer              base=31 n_detections= 17200 n_gt= 17 [7s]


[094.tiff] human breast cancer              base=25 n_detections= 18410 n_gt= 81 [7s]


[201.tiff] canine lung cancer               base=51 n_detections= 15848 n_gt= 17 [4s]


[233.tiff] canine lung cancer               base=25 n_detections= 17614 n_gt= 17 [6s]


[245.tiff] canine lymphosarcoma             base=47 n_detections= 17552 n_gt= 89 [6s]


...

[460.tiff] canine soft tissue sarcoma       base=47 n_detections= 15339 n_gt= 35 [4s]


[529.tiff] human melanoma                   base=37 n_detections= 15862 n_gt= 19 [6s]


[548.tiff] human melanoma                   base=29 n_detections= 17413 n_gt=238 [7s]



14 ROIs x 4 budgets = 56 rows in 83s


,tumor_type,seed_ann_id,base_size,n_gt_mitotic,n_detections,match_radius_px,nms_radius_px,map_median,mad_scale,t_total_s
file_name,,,,,,,,,,
013.tiff,human breast cancer,254,31,17,17200,33.139,33.139,-0.005,0.128,7.4
094.tiff,human breast cancer,2512,25,81,18410,32.630,32.630,-0.003,0.051,6.7
201.tiff,canine lung cancer,4457,51,17,15848,30.222,30.222,-0.028,0.612,4.4
233.tiff,canine lung cancer,5761,25,17,17614,30.222,30.222,-0.016,0.144,6.1
245.tiff,canine lymphosarcoma,6274,47,89,17552,30.222,30.222,-0.007,0.214,6.2
246.tiff,canine lymphosarcoma,6548,41,115,17763,30.222,30.222,-0.006,0.171,6.2
300.tiff,canine cutaneous mast cell tumor,14581,45,180,17940,29.609,29.609,-0.057,0.371,5.4
301.tiff,canine cutaneous mast cell tumor,14969,41,217,17963,29.609,29.609,-0.022,0.204,5.2
402.tiff,human neuroendocrine tumor,20254,29,104,17337,33.139,33.139,-0.056,0.921,5.9


## Checks before any table

Three things must hold for the deep-floor-then-truncate shortcut to be valid, and for the D7
NMS-radius invariant to be more than documentation:

1. Every ROI's deep-floor pool clears the largest budget (K = 50) -- otherwise a
   "precision@50" would silently be computed over fewer than 50 candidates.
2. `budget_delivered == budget` for every row -- the direct restatement of (1) at every K, not
   just the largest.
3. No surviving candidate lies within one match radius of the seed's own (removed) annotation.
   At NMS radius == match radius this must be zero: the self-correlation is the response map's
   global maximum, so NMS keeps it first and suppresses everything within one NMS radius before
   self-hit removal ever deletes it -- "counted, not argued", per the sibling notebook's Gate 3.

`invariants.check_nms_radius` (D7) and `invariants.check_no_cap` were already asserted per ROI
inside `evaluate_arms`, above, and are already sitting in `VERIF`.

In [5]:
assert (ROI['n_detections'] >= max(BUDGETS)).all(), \
    'a deep-floor pool did not clear the largest budget -- see ROI[\'n_detections\']'
assert (RAW['budget_delivered'] == RAW['budget']).all(), \
    'a budget row was starved -- the deep floor did not have enough survivors somewhere'
assert (ROI['n_near_seed_annulus'] == 0).all(), \
    'a candidate survived inside the removed seed\'s match radius -- see n_near_seed_annulus'

VERIF = pd.concat([VERIF, pd.DataFrame([
    dict(check='deep_pool_covers_max_budget', label='ALL',
         passed=bool((ROI['n_detections'] >= max(BUDGETS)).all())),
    dict(check='no_starvation_any_budget', label='ALL',
         passed=bool((RAW['budget_delivered'] == RAW['budget']).all())),
])], ignore_index=True)
VERIF.to_csv(OUT_VERIF, index=False)
print(f'{len(VERIF)} verification records -> {OUT_VERIF}')
print(f"  passed: {int(VERIF['passed'].sum())} / {len(VERIF)}")
assert VERIF['passed'].all(), 'a check failed -- read the verification CSV before any table below'

44 verification records -> ../results/precision_at_k_14roi_gatedseed_verification.csv
  passed: 44 / 44


## Table A -- precision per ROI, at each budget

One row per ROI (14 total), sorted by domain. `n_gt_mitotic` and `n_detections` are descriptive
context, not a recall metric. `z_at_rank_K` is the per-ROI/per-domain robust-z value that
candidate K happens to sit at -- reported because it visibly varies (as flagged going in),
never applied as a threshold.

In [6]:
RAW['precision_at_budget'] = RAW['tp_at_budget'] / RAW['budget_delivered']

order = ROI[['tumor_type']].reset_index().sort_values(['tumor_type', 'file_name'])

rows_a = []
for _, o in order.iterrows():
    fn = o['file_name']
    r = dict(file_name=fn, domain=o['tumor_type'],
             n_gt_mitotic=int(ROI.loc[fn, 'n_gt_mitotic']),
             n_detections=int(ROI.loc[fn, 'n_detections']))
    sub = RAW[RAW['file_name'] == fn].set_index('budget')
    for k in BUDGETS:
        r[f'z_at_rank_{k}'] = round(float(sub.loc[k, 'z_at_rank']), 3)
        r[f'budget_delivered_{k}'] = int(sub.loc[k, 'budget_delivered'])
        r[f'tp_at_{k}'] = int(sub.loc[k, 'tp_at_budget'])
        r[f'precision_at_{k}'] = round(float(sub.loc[k, 'precision_at_budget']), 4)
    rows_a.append(r)

TABLE_A = pd.DataFrame(rows_a)
TABLE_A.to_csv(OUT_PER_ROI, index=False)
print(f'-> {OUT_PER_ROI}  ({len(TABLE_A)} rows)')
TABLE_A

-> ../results/precision_at_k_14roi_gatedseed_per_roi.csv  (14 rows)


,file_name,domain,n_gt_mitotic,n_detections,z_at_rank_10,budget_delivered_10,tp_at_10,precision_at_10,z_at_rank_20,budget_delivered_20,tp_at_20,precision_at_20,z_at_rank_30,budget_delivered_30,tp_at_30,precision_at_30,z_at_rank_50,budget_delivered_50,tp_at_50,precision_at_50
0,300.tiff,canine cutaneous mast cell tumor,180,17940,8.523,10,7,0.7,7.794,20,14,0.70,7.577,30,20,0.6667,7.196,50,32,0.64
1,301.tiff,canine cutaneous mast cell tumor,217,17963,7.078,10,9,0.9,6.830,20,16,0.80,6.634,30,24,0.8000,6.431,50,38,0.76
2,201.tiff,canine lung cancer,17,15848,6.339,10,3,0.3,6.053,20,5,0.25,5.914,30,6,0.2000,5.623,50,11,0.22
3,233.tiff,canine lung cancer,17,17614,10.805,10,4,0.4,10.254,20,7,0.35,10.019,30,7,0.2333,9.462,50,10,0.20
4,245.tiff,canine lymphosarcoma,89,17552,5.281,10,1,0.1,5.125,20,1,0.05,4.994,30,1,0.0333,4.875,50,4,0.08
5,246.tiff,canine lymphosarcoma,115,17763,9.720,10,10,1.0,8.791,20,18,0.90,8.404,30,24,0.8000,7.667,50,32,0.64
6,459.tiff,canine soft tissue sarcoma,130,17994,7.604,10,5,0.5,7.008,20,10,0.50,6.886,30,15,0.5000,6.495,50,24,0.48
7,460.tiff,canine soft tissue sarcoma,35,15339,7.557,10,7,0.7,6.937,20,11,0.55,6.551,30,13,0.4333,6.264,50,17,0.34
8,013.tiff,human breast cancer,17,17200,20.260,10,4,0.4,18.340,20,5,0.25,17.592,30,6,0.2000,16.435,50,9,0.18
9,094.tiff,human breast cancer,81,18410,18.692,10,5,0.5,17.660,20,10,0.50,16.727,30,16,0.5333,15.546,50,25,0.50


## Table B -- precision per domain, at each budget

7 domains x 4 budgets = 28 rows. `precision_pooled = sum(tp_at_budget) / sum(budget_delivered)`
across that domain's 2 ROIs -- algebraically identical to the simple mean of the two ROIs'
`precision_at_K` here, because the checks above guarantee `budget_delivered == K` for both. The
worst-ROI columns name the weaker of the two ROIs per domain/budget, so a bad cell can't hide
behind the pooled number.

In [7]:
# RAW already carries 'tumor_type' (it was passed through `context` into evaluate_arms),
# so this groups it directly rather than re-merging against ROI and colliding column names.
rows_b = []
for (domain, k), g in RAW.groupby(['tumor_type', 'budget']):
    tp_sum = int(g['tp_at_budget'].sum())
    delivered_sum = int(g['budget_delivered'].sum())
    worst = g.loc[g['precision_at_budget'].idxmin()]
    rows_b.append(dict(
        domain=domain, n_roi=len(g), K=int(k),
        tp_sum=tp_sum, delivered_sum=delivered_sum,
        precision_pooled=round(tp_sum / delivered_sum, 4) if delivered_sum else np.nan,
        precision_worst_roi=round(float(worst['precision_at_budget']), 4),
        worst_roi_file=str(worst['file_name']),
    ))

TABLE_B = pd.DataFrame(rows_b).sort_values(['domain', 'K']).reset_index(drop=True)
TABLE_B.to_csv(OUT_BY_DOMAIN, index=False)
print(f'-> {OUT_BY_DOMAIN}  ({len(TABLE_B)} rows = 7 domains x {len(BUDGETS)} budgets)')
TABLE_B

-> ../results/precision_at_k_14roi_gatedseed_by_domain.csv  (28 rows = 7 domains x 4 budgets)


,domain,n_roi,K,tp_sum,delivered_sum,precision_pooled,precision_worst_roi,worst_roi_file
0,canine cutaneous mast cell tumor,2,10,16,20,0.8000,0.7000,300.tiff
1,canine cutaneous mast cell tumor,2,20,30,40,0.7500,0.7000,300.tiff
2,canine cutaneous mast cell tumor,2,30,44,60,0.7333,0.6667,300.tiff
3,canine cutaneous mast cell tumor,2,50,70,100,0.7000,0.6400,300.tiff
4,canine lung cancer,2,10,7,20,0.3500,0.3000,201.tiff
5,canine lung cancer,2,20,12,40,0.3000,0.2500,201.tiff
6,canine lung cancer,2,30,13,60,0.2167,0.2000,201.tiff
7,canine lung cancer,2,50,21,100,0.2100,0.2000,233.tiff
8,canine lymphosarcoma,2,10,11,20,0.5500,0.1000,245.tiff
9,canine lymphosarcoma,2,20,19,40,0.4750,0.0500,245.tiff


## Closing summary

`precision_at_K` (Table A) and `precision_pooled` / `precision_worst_roi` (Table B) are the
deliverable. `compare.evaluate_arms` computes `recall_at_budget`, `full_list_recall` and
`read_50..read_100` as an unavoidable byproduct of reusing that harness -- it's what the module
was built for -- but those columns exist only in `results/precision_at_k_14roi_gatedseed_raw.csv`, for
provenance. They are never selected into Table A or Table B and are not part of this
analysis.

In [8]:
RAW.to_csv(OUT_RAW, index=False)
print(f'-> {OUT_RAW}  ({len(RAW)} rows; recall-family columns retained here only, for provenance)')
print(f'\nnotebook ran in {time.time() - NB_T0:.0f}s')

-> ../results/precision_at_k_14roi_gatedseed_raw.csv  (56 rows; recall-family columns retained here only, for provenance)

notebook ran in 83s
